# EdgeTune: PEFT + Quantization/Pruning Pipeline Walkthrough

Welcome to **EdgeTune**! This narrated Jupyter notebook walks through the full end-to-end pipeline:
1. **Baseline Model Loading & Evaluation** (Qwen2.5-0.5B-Instruct FP16 baseline)
2. **Parameter-Efficient Fine-Tuning** (LoRA & QLoRA on dialogue summarization)
3. **Structured & Unstructured Pruning** (50% sparsity target)
4. **Groupwise 4-Bit Weight Quantization**
5. **Combined Compression Stacking** (Fine-tuned → Pruned → Quantized)
6. **Pareto Frontier Analysis & Metric Tradeoffs**

In [ ]:
import os
import torch
from edgetune.config import load_base_config
from edgetune.model_loader import load_model_and_tokenizer, get_optimal_device, get_model_size_mb
from edgetune.peft_trainer import prepare_samsum_dataset
from edgetune.benchmark import benchmark_model_variant

device = get_optimal_device()
print(f"Active Device: {device.type}")

## 1. Baseline Model Loading & FP16 Benchmark
We establish our FP16 baseline anchor on the SAMSum dialogue summarization dataset.

In [ ]:
base_cfg = load_base_config("../configs/base_model.yaml")
base_model, tokenizer, device = load_model_and_tokenizer(base_cfg["model"])
print(f"Baseline Parameter Size: {get_model_size_mb(base_model)} MB")

## 2. Parameter-Efficient Fine-Tuning (LoRA & QLoRA)
Fine-tuning injects low-rank rank-16 adapters into attention layers while freezing base weights.

In [ ]:
from edgetune.peft_trainer import train_peft_model
from edgetune.config import load_lora_config

lora_cfg = load_lora_config("../configs/lora.yaml")
print("LoRA Target Modules:", lora_cfg["lora"].target_modules)

## 3. Pruning & Quantization Modules
Applying magnitude-based 50% pruning followed by 4-bit uniform groupwise quantization.

In [ ]:
from edgetune.pruner import apply_pruning
from edgetune.quantizer import apply_quantization
from edgetune.config import load_prune_config, load_quant_config

p_cfg = load_prune_config("../configs/pruning.yaml")
q_cfg = load_quant_config("../configs/quantization_gptq.yaml")

print(f"Pruning Sparsity: {p_cfg.sparsity * 100}%")
print(f"Quantization Bits: {q_cfg.bits}-bit (group_size={q_cfg.group_size})")

## 4. Benchmark Tradeoff Visualization
Examine generated Pareto frontier charts comparing ROUGE-L vs Disk Size and Latency vs Memory.

In [ ]:
from IPython.display import Image, display
if os.path.exists("../results/comparison_chart.png"):
    display(Image(filename="../results/comparison_chart.png"))
else:
    print("Run scripts/run_full_benchmark_sweep.py to view visual results.")